In [4]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# A simple 7x7 grayscale image - a bright square on dark background
image = tf.constant([[
    [0., 0., 0., 0., 0., 0., 0.],
    [0., 1., 1., 1., 1., 1., 0.],
    [0., 1., 1., 1., 1., 1., 0.],
    [0., 1., 1., 1., 1., 1., 0.],
    [0., 1., 1., 1., 1., 1., 0.],
    [0., 1., 1., 1., 1., 1., 0.],
    [0., 0., 0., 0., 0., 0., 0.]
]], dtype=tf.float32)

# Reshape to [batch=1, H=7, W=7, channels=1]
image = image[..., tf.newaxis]
print(f"the 7x7 grayscale image setup is : \n{image[0, :, :, 0]}")
print("hello world")

the 7x7 grayscale image setup is : 
[[0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 1. 1. 1. 1. 0.]
 [0. 1. 1. 1. 1. 1. 0.]
 [0. 1. 1. 1. 1. 1. 0.]
 [0. 1. 1. 1. 1. 1. 0.]
 [0. 1. 1. 1. 1. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0.]]
hello world


In [6]:
# ============================================================
# 1. Horizontal edge filter
# ============================================================
# Detects horizontal edges: top row negative, bottom row positive.
# It responds strongly where brightness changes top-to-bottom.
h_filter = tf.constant([
    [[-1.], [-1.], [-1.]],
    [[0.], [0.], [0.]],
    [[1.], [1.], [1.]]
], dtype=tf.float32)
h_filter = h_filter[:, :, tf.newaxis, :] 

h_output = tf.nn.conv2d(image, h_filter, strides=[1, 1, 1, 1], padding='VALID')
print("1. Horizontal edge filter output (VALID, stride=1):")
print(h_output[0, :, :, 0].numpy())


1. Horizontal edge filter output (VALID, stride=1):
[[ 2.  3.  3.  3.  2.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [-2. -3. -3. -3. -2.]]


In [8]:
# ============================================================
# 2. Vertical edge filter
# ============================================================
# Same idea, rotated 90 degrees: detects LEFT/RIGHT brightness changes.
v_filter = tf.constant([
    [[-1.], [0.], [1.]],
    [[-1.], [0.], [1.]],
    [[-1.], [0.], [1.]]
], dtype=tf.float32)
v_filter = v_filter[:, :, tf.newaxis, :]

v_output = tf.nn.conv2d(image, v_filter, strides=[1, 1, 1, 1], padding='VALID')
print("\n2. Vertical edge filter output (VALID, stride=1):")
print(v_output[0, :, :, 0].numpy())



2. Vertical edge filter output (VALID, stride=1):
[[ 2.  0.  0.  0. -2.]
 [ 3.  0.  0.  0. -3.]
 [ 3.  0.  0.  0. -3.]
 [ 3.  0.  0.  0. -3.]
 [ 2.  0.  0.  0. -2.]]


In [10]:
# ============================================================
# 3. Blur filter
# ============================================================
# A 3x3 box-average filter: every value = 1/9. This just replaces each
# pixel with the mean of its 3x3 neighborhood -> smoothing/blurring.
blur_filter = tf.constant([
    [[1/9], [1/9], [1/9]],
    [[1/9], [1/9], [1/9]],
    [[1/9], [1/9], [1/9]]
], dtype=tf.float32)
    
blur_filter = blur_filter[:, :, tf.newaxis, :]

blur_output = tf.nn.conv2d(image, blur_filter, strides=[1, 1, 1, 1], padding='VALID')
print("\n3. Blur filter output (VALID, stride=1):")
print(np.round(blur_output[0, :, :, 0].numpy(), 3))



3. Blur filter output (VALID, stride=1):
[[0.444 0.667 0.667 0.667 0.444]
 [0.667 1.    1.    1.    0.667]
 [0.667 1.    1.    1.    0.667]
 [0.667 1.    1.    1.    0.667]
 [0.444 0.667 0.667 0.667 0.444]]


In [16]:
# ============================================================
# 4. padding='SAME' vs padding='VALID'
# ============================================================
valid_out = tf.nn.conv2d(image, h_filter, strides=[1, 1, 1, 1], padding='VALID')
print(f"\n\nvalid:\n{valid_out[0, :, :, 0]}")
same_out = tf.nn.conv2d(image, h_filter, strides=[1, 1, 1, 1], padding='SAME')
print(f"\n\nsame:\n{same_out[0, :, :, 0]}")

print("\n4. VALID vs SAME output shapes (stride=1, 3x3 kernel, 7x7 input):")
print("VALID output shape:", valid_out.shape)  # (1, 5, 5, 1)
print("SAME  output shape:", same_out.shape)   # (1, 7, 7, 1)




valid:
[[ 2.  3.  3.  3.  2.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [-2. -3. -3. -3. -2.]]


same:
[[ 1.  2.  3.  3.  3.  2.  1.]
 [ 1.  2.  3.  3.  3.  2.  1.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [-1. -2. -3. -3. -3. -2. -1.]
 [-1. -2. -3. -3. -3. -2. -1.]]

4. VALID vs SAME output shapes (stride=1, 3x3 kernel, 7x7 input):
VALID output shape: (1, 5, 5, 1)
SAME  output shape: (1, 7, 7, 1)


In [17]:
# ============================================================
# 5. strides=[1,2,2,1] (stride=2)
# ============================================================
valid_stride2 = tf.nn.conv2d(image, h_filter, strides=[1, 2, 2, 1], padding='VALID')
same_stride2 = tf.nn.conv2d(image, h_filter, strides=[1, 2, 2, 1], padding='SAME')

print("\n5. Stride=2 output shapes:")
print("VALID stride=2 shape:", valid_stride2.shape)  # (1, 3, 3, 1)
print(valid_stride2[0, :, :, 0].numpy())
print("SAME  stride=2 shape:", same_stride2.shape)   # (1, 4, 4, 1)
print(same_stride2[0, :, :, 0].numpy())



5. Stride=2 output shapes:
VALID stride=2 shape: (1, 3, 3, 1)
[[ 2.  3.  2.]
 [ 0.  0.  0.]
 [-2. -3. -2.]]
SAME  stride=2 shape: (1, 4, 4, 1)
[[ 1.  3.  3.  1.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]
 [-1. -3. -3. -1.]]
